# 03.04 → Construção e auditoria das sequências temporais

Compara **24, 48, 168 e 240 horas de histórico × vetor das próximas 24 horas** após o tratamento do `03_03`. Preserva o `03_02`, o pré-processamento e as datas originais. Não treina modelos.

O `03_03` contém uma auditoria demonstrativa 24 × 1. O `04_01` já materializa 240 × 24, mas aceita alvos tratados no treino e avalia validação/teste às 00h. Aqui o critério principal × **horário e idêntico nas três partições: todos os 24 alvos observados**, histórico finito e disponível na emissão, mesmo bloco e mesma partição. As contagens diárias são relatadas separadamente. Os resultados desses protocolos não devem ser confundidos.

As partições cronológicas 70/15/15 são herdadas e **provisórias**. A auditoria de cobertura mensal subsidia sua revisão antes do treinamento. Não há escolha automática de histórico por resultados de teste.

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'reports/tcc/preprocessamento_horario_bak2025/manifesto.json').is_file())
NOTEBOOK = RAIZ / 'notebooks/tcc/03_04_construcao_sequencias_temporais.ipynb'
RELATORIO = RAIZ / 'reports/tcc/03_04_sequencias_temporais'
TENSORES = RAIZ / 'data/processed/sequencias_temporais_24h'
RELATORIO.mkdir(parents=True, exist_ok=True)
TENSORES.mkdir(parents=True, exist_ok=True)
HISTORICOS, HORIZONTE = [24, 48, 168, 240], 24
PARTICOES = ['treino', 'validacao', 'teste']
FEATURES = {
    'australia': ['Global_Horizontal_Radiation', 'Weather_Temperature_Celsius',
                  'Weather_Relative_Humidity', 'Wind_Speed', 'Active_Power'],
    'brasil': ['Active_Power'],
}

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for bloco in iter(lambda: f.read(1024*1024), b''):
            h.update(bloco)
    return h.hexdigest()

proveniencia = RAIZ / 'reports/tcc/preprocessamento_horario_bak2025/manifesto.json'
origem = json.loads(proveniencia.read_text(encoding='utf-8'))
escalas_path = proveniencia.parent / 'tabelas/escalas_treino.csv'
capacidades = pd.read_csv(escalas_path).set_index('dominio').capacidade_ac_kw.to_dict()
fontes = {str(p.relative_to(RAIZ)): sha256(p) for p in [proveniencia, escalas_path]}
bases = {}
for dominio in FEATURES:
    item = origem['exportados'][dominio+'_todos_blocos']
    path = RAIZ / item['arquivo']
    if sha256(path) != item['sha256']:
        raise ValueError('Base diverge do manifesto do 03_03: '+str(path))
    fontes[item['arquivo']] = item['sha256']
    d = pd.read_csv(path, index_col='timestamp', parse_dates=True)
    d['disponivel_em'] = pd.to_datetime(d.disponivel_em, errors='raise')
    bases[dominio] = d

## Continuidade, disponibilidade e qualidade

Para emissão em `t`, a entrada vai de `t−L` a `t−1h` e o alvo de `t` a `t+23h`. São necessárias `L+24` horas consecutivas: 48, 72, 192 ou 264 horas. Não se concatenam blocos separados por lacunas, nem se empresta histórico da partição anterior.

A média rotulada hh:00 só fica disponível ao terminar a hora. Uma interpolação só pode entrar após a disponibilidade de sua borda direita. Todos os alvos precisam ter `potencia_observada_horaria=True`. Esse indicador não garante cobertura intrahorária completa ou oito inversores: as máscaras e contagens acompanham os tensores.

As funções de validação/indexação e normalização abaixo preservam o contrato implementado no `04_01`; a seleção estrita e a materialização para vários históricos são específicas desta comparação.

In [2]:
def validar_base_janelas(dados, colunas):
    exigidas = set(colunas) | {"Active_Power", "bloco_id", "particao_proposta",
                              "disponivel_em", "potencia_observada_horaria", "n_5min_Active_Power"}
    if exigidas-set(dados.columns):
        raise ValueError(f"Colunas ausentes: {exigidas-set(dados.columns)}")
    if not isinstance(dados.index, pd.DatetimeIndex) or not dados.index.is_unique or not dados.index.is_monotonic_increasing:
        raise ValueError("Timestamps precisam ser únicos e ordenados.")
    if dados.empty or not np.isfinite(dados[colunas].to_numpy(dtype=float)).all():
        raise ValueError("Entradas precisam ser finitas e não vazias.")
    if dados.disponivel_em.isna().any() or dados.potencia_observada_horaria.isna().any():
        raise ValueError("Máscaras de qualidade/disponibilidade incompletas.")
    if not pd.api.types.is_bool_dtype(dados.potencia_observada_horaria):
        raise ValueError("Máscara observada precisa ter tipo booleano.")
    if not dados.bloco_id.gt(0).all():
        raise ValueError("A base compacta só deve conter blocos positivos.")
    ordem = dados.particao_proposta.map({"treino":0, "validacao":1, "teste":2})
    if ordem.isna().any() or ordem.diff().dropna().lt(0).any():
        raise ValueError("Partições precisam respeitar ordem cronológica.")
    for _, grupo in dados.groupby("bloco_id", sort=False):
        if not grupo.index.to_series().diff().dropna().eq(pd.Timedelta(hours=1)).all():
            raise ValueError("Um bloco contém lacuna ou frequência diferente de uma hora.")


def indexar_janelas_multiplos_passos(dados, colunas, historico=240, saida=24):
    """Audita candidatos de passo 1h; mantém índices globais sem unir blocos."""
    if historico<1 or saida<1:
        raise ValueError("Histórico e saída precisam ser positivos.")
    validar_base_janelas(dados, colunas)
    pos_global = pd.Series(np.arange(len(dados)), index=dados.index)
    registros, segmentos = [], []
    for (bloco, part), g in dados.groupby(["bloco_id", "particao_proposta"], sort=False):
        segmentos.append(dict(bloco_id=int(bloco), particao=part, inicio=g.index[0], fim=g.index[-1],
            horas=len(g), horas_minimas=historico+saida,
            candidatos_teoricos=max(0,len(g)-historico-saida+1)))
        if len(g)<historico+saida:
            continue
        p = np.arange(historico, len(g)-saida+1)  # primeira posição do vetor-alvo
        pos = pos_global.loc[g.index].to_numpy()
        emissao = g.index[p]
        av = g.disponivel_em.to_numpy(dtype="datetime64[ns]")
        maior_disp = np.lib.stride_tricks.sliding_window_view(av,historico).max(axis=1)[:len(p)]
        observado = g.potencia_observada_horaria.to_numpy(dtype=bool)
        observados_y = np.lib.stride_tricks.sliding_window_view(observado,saida).sum(axis=1)[p]
        registros.append(pd.DataFrame(dict(
            bloco_id=int(bloco), particao=part, inicio_hist_pos=pos[p-historico],
            inicio_alvo_pos=pos[p], fim_alvo_pos=pos[p+saida-1],
            inicio_historico=g.index[p-historico], fim_historico=g.index[p-1],
            emissao=emissao, inicio_alvo=g.index[p], fim_alvo=g.index[p+saida-1],
            ultima_disponibilidade=maior_disp,
            historico_disponivel=maior_disp<=emissao.to_numpy(),
            emissao_diaria=(emissao.hour==0)&(emissao.minute==0),
            horas_alvo_observadas=observados_y,
            saida_total_observada=observados_y==saida)))
    campos = ["bloco_id","particao","inicio_hist_pos","inicio_alvo_pos","fim_alvo_pos",
              "inicio_historico","fim_historico","emissao","inicio_alvo","fim_alvo",
              "ultima_disponibilidade","historico_disponivel","emissao_diaria",
              "horas_alvo_observadas","saida_total_observada"]
    indice = pd.concat(registros,ignore_index=True)[campos] if registros else pd.DataFrame(columns=campos)
    indice.insert(0,"janela_id",np.arange(len(indice)))
    return indice, pd.DataFrame(segmentos)

In [3]:
def selecionar_observadas(indice):
    mascara = indice.historico_disponivel.astype(bool) & indice.saida_total_observada.astype(bool)
    return indice.loc[mascara].copy()

indices, selecionadas, segmentos, linhas = {}, {}, [], []
for dominio, dados in bases.items():
    for historico in HISTORICOS:
        idx, seg = indexar_janelas_multiplos_passos(dados, FEATURES[dominio], historico, HORIZONTE)
        sel = selecionar_observadas(idx)
        indices[dominio, historico], selecionadas[dominio, historico] = idx, sel
        segmentos.append(seg.assign(dominio=dominio, historico_horas=historico))
        sel.to_csv(RELATORIO/f'{dominio}_L{historico}_indice.csv', index=False)
        for part in PARTICOES:
            g = idx.loc[idx.particao.eq(part)]
            disp = g.historico_disponivel.astype(bool)
            obs = g.saida_total_observada.astype(bool)
            uso = sel.loc[sel.particao.eq(part)]
            linhas.append(dict(dominio=dominio, historico_horas=historico, particao=part,
                candidatas=len(g), rejeitadas_disponibilidade=int((~disp).sum()),
                rejeitadas_alvos_apos_disponibilidade=int((disp & ~obs).sum()),
                janelas_observadas=len(uso), emissoes_00h=int(uso.emissao_diaria.sum())))
contagens = pd.DataFrame(linhas)
segmentos = pd.concat(segmentos, ignore_index=True)
contagens.to_csv(RELATORIO/'contagens.csv', index=False)
segmentos.to_csv(RELATORIO/'segmentos.csv', index=False)
display(contagens.pivot(index=['dominio','historico_horas'], columns='particao', values='janelas_observadas').reindex(columns=PARTICOES))
display(contagens)

particao                   treino  validacao  teste
dominio   historico_horas                          
australia 24                 5486       2601   2275
          48                 5462       2434   2159
          168                5303       1787   1662
          240                5160       1503   1476
brasil    24                 3269        827    662
          48                 2945        779    588
          168                1682        539    290
          240                1162        395    193

,dominio,historico_horas,particao,candidatas,rejeitadas_disponibilidade,rejeitadas_alvos_apos_disponibilidade,janelas_observadas,emissoes_00h
0,australia,24,treino,18297,4,12807,5486,228
1,australia,24,validacao,3677,2,1074,2601,110
2,australia,24,teste,3418,8,1135,2275,99
3,australia,48,treino,18105,3,12640,5462,227
4,australia,48,validacao,3509,2,1073,2434,103
5,australia,48,teste,3177,8,1010,2159,93
6,australia,168,treino,17311,3,12005,5303,221
7,australia,168,validacao,2669,1,881,1787,76
8,australia,168,teste,2284,6,616,1662,70
9,australia,240,treino,16879,3,11716,5160,215


## Emissões comuns e cobertura sazonal

Uma comparação de modelos com históricos diferentes precisa usar as mesmas emissões de avaliação. Exporta-se a interseção das emissões válidas entre os quatro históricos, por domínio e partição; não se exige coincidência de datas entre países. As contagens individuais acima medem aproveitamento, e não desempenho comparável entre conjuntos diferentes. Não se equilibram estações removendo ou movendo dados automaticamente.

In [4]:
comuns, cobertura, parciais = [], [], []
for dominio, dados in bases.items():
    for part in PARTICOES:
        conjuntos = [pd.DatetimeIndex(selecionadas[dominio, h].loc[
            selecionadas[dominio, h].particao.eq(part), 'emissao']) for h in HISTORICOS]
        inter = conjuntos[0]
        for conjunto in conjuntos[1:]:
            inter = inter.intersection(conjunto)
        comuns.append(pd.DataFrame(dict(dominio=dominio, particao=part, emissao=inter)))
    d = dados.assign(mes=dados.index.to_period('M').astype(str))
    for (part, mes), g in d.groupby(['particao_proposta', 'mes'], sort=False):
        cobertura.append(dict(dominio=dominio, particao=part, mes=mes,
            horas_elegiveis=len(g), horas_observadas=int(g.potencia_observada_horaria.sum())))
    for h in HISTORICOS:
        s = selecionadas[dominio, h].copy()
        s['mes'] = pd.to_datetime(s.emissao).dt.to_period('M').astype(str)
        s.groupby(['particao','mes']).size().rename('janelas').reset_index().to_csv(
            RELATORIO/f'{dominio}_L{h}_cobertura_emissoes.csv', index=False)
    if dominio == 'brasil':
        for part, g in dados.groupby('particao_proposta', sort=False):
            parciais.append(dict(particao=part, horas=len(g),
                horas_com_soma_parcial=int(g.n_leituras_soma_parcial.gt(0).sum()),
                horas_com_12_leituras_oito_inversores=int(g.n_leituras_oito_inversores.eq(12).sum())))
comuns = pd.concat(comuns, ignore_index=True)
comuns.to_csv(RELATORIO/'emissoes_comuns.csv', index=False)
pd.DataFrame(cobertura).to_csv(RELATORIO/'cobertura_mensal_particoes.csv', index=False)
pd.DataFrame(parciais).to_csv(RELATORIO/'cobertura_inversores_brasil.csv', index=False)
display(comuns.groupby(['dominio','particao']).size().rename('emissoes_comuns'))
display(pd.DataFrame(cobertura))
display(pd.DataFrame(parciais))

# Disponibilidade bruta horária por canal e ano, antes da seleção simultânea/QC.
path = proveniencia.parent / 'tabelas/australia_contagens_5min.csv'
fontes[str(path.relative_to(RAIZ))] = sha256(path)
cont = pd.read_csv(path, index_col='timestamp', parse_dates=True)[FEATURES['australia']]
canal_ano = cont.gt(0).groupby(cont.index.year).sum().rename_axis('ano')
canal_ano['horas_calendario'] = cont.groupby(cont.index.year).size()
canal_ano['horas_todos_canais_com_leitura'] = cont.gt(0).all(axis=1).groupby(cont.index.year).sum()
canal_ano.to_csv(RELATORIO/'australia_disponibilidade_canal_ano.csv')
display(canal_ano)

dominio    particao 
australia  teste        1476
           treino       5160
           validacao    1503
brasil     teste         193
           treino       1162
           validacao     395
Name: emissoes_comuns, dtype: int64

,dominio,particao,mes,horas_elegiveis,horas_observadas
0,australia,treino,2013-07,144,144
1,australia,treino,2013-08,744,726
2,australia,treino,2013-09,720,688
3,australia,treino,2013-10,744,711
4,australia,treino,2013-11,720,674
5,australia,treino,2013-12,744,714
6,australia,treino,2014-01,744,716
7,australia,treino,2014-02,672,640
8,australia,treino,2014-03,744,698
9,australia,treino,2014-04,696,655


,particao,horas,horas_com_soma_parcial,horas_com_12_leituras_oito_inversores
0,treino,4417,201,1890
1,validacao,946,23,356
2,teste,947,19,361


,Global_Horizontal_Radiation,Weather_Temperature_Celsius,Weather_Relative_Humidity,Wind_Speed,Active_Power,horas_calendario,horas_todos_canais_com_leitura
ano,,,,,,,
2013,3815,3815,3815,3815,3815,3816,3815
2014,8760,8760,8760,8760,8760,8760,8760
2015,8760,8760,8760,8760,8760,8760,8760
2016,8784,8784,8784,7070,8784,8784,7070
2017,8711,8711,8711,0,8711,8760,0
2018,8760,8760,8760,0,8760,8760,0
2019,8580,8580,8580,0,8580,8760,0
2020,8783,8783,8783,0,8783,8784,0
2021,8384,8384,8384,0,8384,8760,0


## Tensores e escalas

Exportam-se 24 arquivos NPZ: dois domínios × quatro históricos × três partições. `X=(N,L,C)`, `X_potencia=(N,L,1)` e `y=(N,24)`. `y_kw` conserva a escala física para permitir revisar as capacidades. Arquivos sem amostras mantém essas dimensões com `N=0`.

AP usa as capacidades **adotadas, ainda não confirmadas**, de 24 kW e 1000 kW. Meteorologia usa z-score ajustado exclusivamente no treino; as mesmas escalas servem aos quatro históricos. Não se utiliza meteorologia futura. As máscaras de interpolação, timestamps e contagens de contribuintes permanecem disponíveis. A unidade original do vento continua pendente de confirmação.

In [5]:
def normalizar_entradas(dados, colunas, capacidade):
    if capacidade<=0 or not np.isfinite(capacidade):
        raise ValueError("Capacidade precisa ser finita e positiva.")
    treino = dados.particao_proposta.eq("treino")
    fim = dados.index[treino][-1]+pd.Timedelta(hours=1)
    usar = treino & dados.disponivel_em.le(fim)
    if not usar.any(): raise ValueError("Sem horas elegíveis para ajustar escalas.")
    matriz=pd.DataFrame(index=dados.index)
    parametros=[]
    for c in colunas:
        if c=="Active_Power":
            matriz[c]=dados[c]/capacidade
            parametros.append(dict(variavel=c,metodo="pu_capacidade",media=0.,escala=capacidade,n_ajuste=0))
        else:
            media=float(dados.loc[usar,c].mean())
            desvio=float(dados.loc[usar,c].std(ddof=0))
            desvio=desvio if desvio>1e-8 else 1.
            matriz[c]=(dados[c]-media)/desvio
            parametros.append(dict(variavel=c,metodo="z_score_treino",media=media,escala=desvio,n_ajuste=int(usar.sum())))
    return matriz.to_numpy(dtype=np.float32), pd.DataFrame(parametros)

In [6]:
def materializar_sequencias(dados, indice, matriz, colunas, capacidade, historico, saida=24):
    # Indexação explícita funciona também para N=0 e bases menores que L+H.
    px = indice.inicio_hist_pos.to_numpy(dtype=int)[:,None] + np.arange(historico)
    py = indice.inicio_alvo_pos.to_numpy(dtype=int)[:,None] + np.arange(saida)
    x = matriz[px].copy()
    im = dados[['interpolado_'+c for c in colunas]].to_numpy(dtype=bool)
    y_kw = dados.Active_Power.to_numpy(dtype=np.float32)[py]
    ap = colunas.index('Active_Power')
    pacote = dict(X=x, X_potencia=x[:,:,ap:ap+1].copy(), y=y_kw/np.float32(capacidade), y_kw=y_kw,
        mask_X_interpolado=im[px], mask_y_observado=dados.potencia_observada_horaria.to_numpy(dtype=bool)[py],
        timestamps_X=dados.index.to_numpy()[px], timestamps_y=dados.index.to_numpy()[py],
        emissao=indice.emissao.to_numpy(dtype='datetime64[ns]'),
        bloco_id=indice.bloco_id.to_numpy(dtype=np.int64),
        features=np.asarray(colunas, dtype='U48'), capacidade_kw=np.asarray(capacidade),
        historico_horas=np.asarray(historico), saida_horas=np.asarray(saida))
    for c in ['n_5min_Active_Power','n_leituras_oito_inversores','n_leituras_soma_parcial']:
        if c in dados:
            pacote['y_'+c] = dados[c].to_numpy()[py]
    return pacote

exportados, parametros = [], []
for dominio, dados in bases.items():
    matriz, params = normalizar_entradas(dados, FEATURES[dominio], capacidades[dominio])
    parametros.append(params.assign(dominio=dominio))
    for historico in HISTORICOS:
        for part in PARTICOES:
            idx = selecionadas[dominio, historico]
            idx = idx.loc[idx.particao.eq(part)]
            pacote = materializar_sequencias(dados, idx, matriz, FEATURES[dominio], capacidades[dominio], historico, HORIZONTE)
            path = TENSORES/f'{dominio}_{part}_L{historico}_H{HORIZONTE}.npz'
            np.savez_compressed(path, **pacote)
            with np.load(path, allow_pickle=False) as z:
                assert z['X'].shape == (len(idx), historico, len(FEATURES[dominio]))
                assert z['y'].shape == (len(idx), HORIZONTE)
                assert np.isfinite(z['X']).all() and np.isfinite(z['y']).all()
                assert z['mask_y_observado'].all()
                assert np.all(np.diff(z['timestamps_X'],axis=1) == np.timedelta64(1,'h'))
                assert np.all(np.diff(z['timestamps_y'],axis=1) == np.timedelta64(1,'h'))
                assert np.all(z['timestamps_X'][:,-1]+np.timedelta64(1,'h') == z['emissao'])
                assert np.all(z['timestamps_y'][:,0] == z['emissao'])
                assert np.allclose(z['y']*capacidades[dominio], z['y_kw'], rtol=1e-6, atol=1e-4)
                for timestamps in [z['timestamps_X'], z['timestamps_y']]:
                    pos = dados.index.get_indexer(timestamps.reshape(-1))
                    assert (pos >= 0).all()
                    assert np.all(dados.bloco_id.to_numpy()[pos].reshape(timestamps.shape) == z['bloco_id'][:,None])
                    assert np.all(dados.particao_proposta.to_numpy()[pos] == part)
            exportados.append(dict(arquivo=str(path.relative_to(RAIZ)), sha256=sha256(path),
                dominio=dominio, particao=part, historico_horas=historico, amostras=len(idx)))
            del pacote
pd.concat(parametros, ignore_index=True).to_csv(RELATORIO/'escalas_treino.csv', index=False)

## Decisões antes da modelagem

A cobertura anual por canal identifica restrições de disponibilidade; não estabelece por si só a causa física de falhas do vento nem substitui os metadados da estação. Os indicadores de somas parciais não corrigem a potência ausente dos inversores. A confirmação documental das capacidades permanece necessária.

A comparação de contagens permite avaliar o custo de entradas longas. A decisão final de períodos deve considerar duração e cobertura sazonal das emissões brasileiras de validação/teste. Não se declara uma divisão definitiva apenas por respeitar 70/15/15. As emissões comuns permitem comparar históricos sem mudar o conjunto avaliado, embora suas janelas se sobreponham e não sejam amostras independentes.

Pré-treinamento, TCN/DC-TCN, destilação e fine-tuning pertencem × próxima etapa. O `04_01` permanece como protocolo anterior específico de 240 horas; seus tensores e relatórios não são substituídos por este notebook.

In [7]:
for relativo, esperado in fontes.items():
    assert sha256(RAIZ/relativo) == esperado, 'Fonte modificada durante a execução.'
codigo = '\n\n'.join(''.join(c['source']) for c in json.loads(NOTEBOOK.read_text(encoding='utf-8'))['cells'] if c['cell_type']=='code')
manifesto = dict(historicos=HISTORICOS, horizonte=HORIZONTE, features=FEATURES,
    selecao='Horária; 24 alvos observados em todas as partições; histórico disponível na emissão',
    particoes='70/15/15 cronológicas herdadas, provisórias; histórico e alvo na mesma partição e bloco',
    capacidades_adotadas_kw=capacidades, capacidades_confirmadas=False,
    fontes_sha256=fontes, codigo_sha256=hashlib.sha256(codigo.encode()).hexdigest(),
    tensores=exportados, contagens=contagens.to_dict(orient='records'))
(RELATORIO/'manifesto.json').write_text(json.dumps(manifesto,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print('Tensores verificados:', len(exportados))
print('Relatório:', RELATORIO)

Tensores verificados: 24
Relatório: C:\Users\camil\Documents\transfer-learning-energy-forecast-tcc\reports\tcc\03_04_sequencias_temporais
